#### **Step 1: Import Required Libraries**


In [20]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp, row_number)
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 22, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [21]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/postgresql/raw/transactions"
df_cleaned_transactions = spark.read.format("parquet").load(raw_transaction_path)
display(df_cleaned_transactions.limit(10))

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 23, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ed6bf6c1-f2fa-403a-9ff2-8b418f584eb3)

### **Column-by-Column Transformation Plan**
txn_id
- Current: Already in consistent custom format TSC########.
- Action: No transformation needed.
- Final: Keep as-is.

user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

#### **Step 3: Inspect the transactions dataframe for data quality checks**

In [22]:
# Select specific columns and count nulls
null_counts = df_cleaned_transactions.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['txn_type', 'payment_method', 'amount', 'status']
    ]
)

# Display results
null_counts.show()

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 24, Finished, Available, Finished)

+--------+--------------+------+------+
|txn_type|payment_method|amount|status|
+--------+--------------+------+------+
|       0|             0|     0|     0|
+--------+--------------+------+------+



#### **Step 4: Data Transformation to txn_type column**
Issues:
- Inconsistent casing and abbreviations (Deposit, deposit, Wd, withdraw, Withdrawal, '')

Transformations:
- Normalize values to title case: 'Deposit' or 'Withdrawal'
- Map variants:'Wd', 'withdraw', 'Withdrawal' → 'Withdrawal', 'deposit' → 'Deposit'
- Replace empty strings with 'Unknown' or NULL (preferably NULL for BI tools)


In [23]:
# Check distinct values in columns txn_type
# df_cleaned_transactions.select("txn_type").distinct().show()

# Clean txn_type column
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "txn_type",
# Then normalize and transform valid values
    when(lower(trim(col("txn_type"))).isin("withdraw", "wd", "withdrawal"), "Withdrawal")
    .when(lower(trim(col("txn_type"))) == "deposit", "Deposit")
# Finally, set anything else to NULL
    .otherwise(None)
)

display(df_cleaned_transactions.select("txn_type").distinct().show())

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 25, Finished, Available, Finished)

+----------+
|  txn_type|
+----------+
|   Deposit|
|Withdrawal|
|      NULL|
+----------+



#### **Step 5: Data Transformation to payment_column column**
Issues:
- Inconsistent capitalization: mpesa, PayPal, Mobile Money, Crypto, Cash, etc.
- Missing values (some entries are empty or None)

Transformations:
- Title-case all non-null entries (e.g., mpesa → Mpesa)
- Replace empty values with 'Unknown' or NULL
- Standardize all options to expected values: ['Bank Transfer', 'Credit Card', 'PayPal', 'Crypto', 'Cash', 'Mobile Money', 'Mpesa']
- Optional: classify unrecognized values as 'Other'


In [24]:
# Check distinct values in column payment_method
# df_cleaned_transactions.select("payment_method").distinct().show()

# Clean payment_method column
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "payment_method",
# Then normalize and transform valid values
    when(lower(trim(col("payment_method"))) == "mpesa", "Mpesa")
    .when(lower(trim(col("payment_method"))) == "deposit", "Deposit")
    .when(lower(trim(col("payment_method"))) == "credit card", "Credit Card")
    .when(lower(trim(col("payment_method"))) == "bank transfer", "Bank Transfer")
    .when(lower(trim(col("payment_method"))) == "crypto", "Crypto")
    .when(lower(trim(col("payment_method"))) == "paypal", "PayPal")
    .when(lower(trim(col("payment_method"))) == "cash", "Cash")
    .when(lower(trim(col("payment_method"))) == "mobile money", "Mobile Money")
# Finally, set anything else to NULL
    .otherwise("Other")
)

display(df_cleaned_transactions.select("payment_method").distinct().show())

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 26, Finished, Available, Finished)

+--------------+
|payment_method|
+--------------+
|   Credit Card|
| Bank Transfer|
|        Crypto|
|         Mpesa|
|         Other|
|        PayPal|
|          Cash|
|  Mobile Money|
+--------------+



#### **Step 6: Data Transformation to status column**
Issues:
- Mixed casing and spelling: Completed, SUCCESS, cancelled, processing, ''

Transformations:
- Normalize to lower or title case (Title Case preferred for display)
- Map values to consistent statuses: ['Completed', 'Pending', 'Failed', 'Cancelled', 'Processing']
- Replace empty values with NULL

In [25]:
# Check distinct values in column status
# df_cleaned_transactions.select("status").distinct().show()

# Clean payment_method column
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "status",
# Then normalize and transform valid values
    when(lower(trim(col("status"))).isin("Completed", "SUCCESS"), "Completed")
    .when(lower(trim(col("status"))) == "cancelled", "Cancelled")
    .when(lower(trim(col("status"))) == "failed", "Failed")
    .when(lower(trim(col("status"))) == "pending", "Pending")
    .when(lower(trim(col("status"))) == "processing", "Processing")
# Finally, set anything else to NULL
    .otherwise(None)
)

display(df_cleaned_transactions.select("status").distinct().show())

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 27, Finished, Available, Finished)

+----------+
|    status|
+----------+
|Processing|
| Cancelled|
|    Failed|
|   Pending|
|      NULL|
+----------+



#### **Step 7: Data Transformation to amount column**
Issues:
- Appears as scientific notation, often close to zero due to formatting issue.
- Some entries are 0.0 to simulate missing/invalid values.

Transformations:
- Convert to float with 2 decimal places.
- Replace 0.0 with NULL to signify missing values.
- Ensure all amounts are numeric (cast if needed).

Note: Decide whether negative amounts are allowed for withdrawals.

In [26]:
# Step 1: Cast to float and round to 2 decimal places with better handling
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "amount",
    bround(col("amount").cast("double"), 2)
)

# Step 2: Replace 0.0 with NULL
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "amount",
    when(col("amount") == 0.0, None).otherwise(col("amount"))
)

# Step 3: Invalidate negative amounts for withdrawals
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "amount",
    when((col("txn_type").rlike("(?i)^Withdrawal")) & (col("amount") < 0), None)
    .otherwise(col("amount"))
)

# Final view
display(df_cleaned_transactions.select("amount").limit(10))


StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 28, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 4bac3177-918d-465c-ba0d-48bb3d2ee2b1)

#### **Step 8: Data Transformation to txn_time column**
Issues:
- Multiple inconsistent formats (6+ variations)
- Could lead to parsing errors during transformation

Transformations:
- Parse string to standardized TIMESTAMP format
- Use multi-format parsing (PySpark or Pandas with try/except)
- Convert all to ISO format or UTC timezone if needed

In [27]:
# First check current distinct values
# print("Original txn_time values:")
# df_cleaned_transactions.select("txn_time").distinct().show(truncate=False)

# Parse multiple date formats and standardize to date-only
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "txn_date",
    coalesce(
        to_date("txn_time", "MMM dd, yyyy HH:mm"),    # Mar 24, 2025 21:22
        to_date("txn_time", "dd MMMM yyyy, HH:mm"),   # 25 February 2025, 18:45
        to_date("txn_time", "dd/MM/yyyy HH:mm"),       # 28/09/2024 02:54
        to_date("txn_time", "yyyy-MM-dd HH:mm:ss"),    # 2024-07-28 12:42:13
        to_date("txn_time", "MM-dd-yyyy hh:mm a"),     # 06-25-2024 05:00 PM
        to_date("txn_time", "yyyy/MM/dd HH:mm:ss"),    # 2024/10/24 15:24:00
        to_date("txn_time", "MMM dd, yyyy"),          # Oct 01, 2024
        to_date("txn_time", "yyyy-MM-dd"),             # 2024-10-24
        to_date("txn_time", "MM/dd/yyyy")             # 07/14/2024
    )
)

# For any remaining unparsed dates, attempt one last cleanup
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "txn_date",
    coalesce(
        col("txn_date"),
        to_date(regexp_replace(col("txn_time"), "([0-9]{2}) ([A-Za-z]{3}) ([0-9]{4})", "$2 $1, $3")),  # 25 Jun 2024 → Jun 25, 2024
        to_date(regexp_replace(col("txn_time"), "([0-9]{2})-([0-9]{2})-([0-9]{4})", "$2/$1/$3"))       # 25-06-2024 → 06/25/2024
    )
)

# Format the final date consistently
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "txn_date",
    date_format(col("txn_date"), "yyyy-MM-dd")
)

# Drop the original txn_time column
df_cleaned_transactions = df_cleaned_transactions.drop("txn_time")

# View final cleaned values
print("Standardized txn_date values:")
df_cleaned_transactions.select("txn_date").distinct().show(truncate=False)

# Show sample of final data
display(df_cleaned_transactions.limit(5))

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 29, Finished, Available, Finished)

Standardized txn_date values:
+----------+
|txn_date  |
+----------+
|2024-09-15|
|2024-07-14|
|2024-08-20|
|2024-10-24|
|2025-02-25|
|2024-08-06|
|2024-10-22|
|2024-12-12|
|2025-04-05|
|2025-03-20|
|2025-01-03|
|2024-07-12|
|2024-10-10|
|2024-12-09|
|2024-11-20|
|2024-12-03|
|2024-09-21|
|2024-11-08|
|2024-11-30|
|2025-04-12|
+----------+
only showing top 20 rows



SynapseWidget(Synapse.DataFrame, 1dadcc3f-2a14-4c6a-b6f1-cf67957f03ac)

#### **Step 9: Data Deduplication on txn_id Column**

Issue:
- Duplicate transactions were detected with the same txn_id, leading to inflated metrics and potential inconsistencies in analytics.

Transformation:
- Applied deduplication by:
- Partitioning the data on txn_id.
- Ordering each partition by txn_time in descending order to retain the most recent entry.
- Assigned row numbers using a window function.
- Filtered to keep only the first (latest) record per txn_id.

Outcome:
- Ensured that each transaction ID is uniquely represented.
- Preserved the most relevant version of each transaction based on the txn_time.

In [28]:
# Add load_timestamp in the format yyyyMMddHHmmss
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "load_timestamp", lit(datetime.now().strftime("%Y%m%d%H%M%S"))
)

# Define a window partitioned by txn_id and ordered by load_timestamp descending
window_spec = Window.partitionBy("txn_id").orderBy(col("load_timestamp").desc())

# Assign row numbers to identify duplicates
df_cleaned_transactions = df_cleaned_transactions.withColumn("row_num", row_number().over(window_spec))

# Filter to keep only the first record per txn_id
df_cleaned_transactions = df_cleaned_transactions.filter(col("row_num") == 1).drop("row_num")

# Show the result (optional)
df_cleaned_transactions.show(10, truncate=False)


StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 30, Finished, Available, Finished)

+-----------+----------+----------+-------+--------------+---------+----------+--------------+
|txn_id     |user_id   |txn_type  |amount |payment_method|status   |txn_date  |load_timestamp|
+-----------+----------+----------+-------+--------------+---------+----------+--------------+
|TSC10000008|BB10004563|Withdrawal|4198.54|Bank Transfer |NULL     |2024-06-07|20250708035849|
|TSC10000015|BB10004003|Deposit   |1751.2 |Cash          |Pending  |2024-12-30|20250708035849|
|TSC10000016|BB10003982|Deposit   |2616.8 |PayPal        |Cancelled|2024-11-22|20250708035849|
|TSC10000021|BB10003532|Deposit   |2281.83|Bank Transfer |NULL     |2025-02-25|20250708035849|
|TSC10000049|BB10000874|Withdrawal|4000.96|Other         |Pending  |2024-12-25|20250708035849|
|TSC10000052|BB10003970|NULL      |2143.35|Cash          |Failed   |2024-07-22|20250708035849|
|TSC10000054|BB10001695|Withdrawal|3825.96|Bank Transfer |NULL     |2025-04-14|20250708035849|
|TSC10000056|BB10004000|Withdrawal|2376.77|Mpesa  

#### **Step 10: Data Load to Silver Lakehouse Layer**
This script processes cleaned transaction data and writes it to the Silver layer in Delta format. It performs incremental loading using a MERGE strategy to avoid duplicates and ensure upserts (insert new records only).

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/postgresql/cleaned/cleaned_transactions.

Prepare Columns for Consistency:
- txn_date: Ensures a proper date column is available for partitioning.
- load_timestamp: Adds a unique timestamp (once per job) for tracking the data load time.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key txn_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by txn_date for efficient querying and file organization.

In [29]:
# Define paths
silver_path = "Files/cdp_silver/postgresql/cleaned/cleaned_transactions"

# Add date column for partitioning (based on txn_timestamp if available)
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "txn_date", to_date("txn_date")  
)

# Add a consistent load timestamp column (once per job run)
load_ts = datetime.now().strftime("%Y%m%d%H%M%S")
df_cleaned_transactions = df_cleaned_transactions.withColumn(
    "load_timestamp", lit(load_ts)
)

# Perform MERGE if Delta table exists
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("target").merge(
        df_cleaned_transactions.alias("source"),
        "target.txn_id = source.txn_id"  # Replace with your unique key
    ).whenNotMatchedInsertAll().execute()

else:
    # First write: create partitioned Delta table
    df_cleaned_transactions.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("txn_date") \
        .save(silver_path)

StatementMeta(, fe6ca75e-bd1a-441e-947c-45cc8ad5bda2, 31, Finished, Available, Finished)